In [0]:
import pandas as pd
import json
import time
from datetime import datetime
import os

csv_path = "/Workspace/Users/kkarwa04@gmail.com/finguard_project/fraud_watchlist_file_generator/fraud_watchlist.csv"
df = pd.read_csv(csv_path)

df['entity_id'] = df['entity_id'].astype(str)
output_path = "/Volumes/finguard/source/fraud_watchlist/source_data"

dbutils.fs.mkdirs(output_path)

latest_watchlist_id = None
try:
    existing_files = dbutils.fs.ls(output_path)
    if existing_files:
        print(f"Found {len(existing_files)} existing files in {output_path}")
        print("Scanning for latest watchlist_id...")

        max_id_num = 0
        for file_info in existing_files:
            if file.info.name.endswith('.json'):
                try:
                    content = dbutils.fs.head(file_info.path, 10000)
                    data = json.loads(content)
                    watchlist_id = data.get('watchlist_id')
                    if watchlist_id.startswith('WL'):
                        id_num = int(watchlist_id[2:])
                        if id_num > max_id_num:
                            max_id_num = id_num
                            latest_watchlist_id = file_info
                except Exception as e:
                    continue

        if latest_watchlist_id:
            print(f"Latest watchlist_id found: {latest_watchlist_id} (numeric: {max_id_num})")
            df["_id_num"] = df["watchlist_id"].str[2:].astype(int)
            df = df[df["_id_num"] > max_id_num].drop(columns=["_id_num"]).reset_index(drop=True)
            print(f"Resuming from next record. {len(df)} rows remaining to process.\n")
        else:
            print("No watchlist_id found in existing files. Starting from the beginning.\n")
    else:
        print("No existing files found. Starting from the beginning.\n")
except Exception as e:
    print(f"Could not read existing files: {e}")
    print("Starting from the beginning.\n")

if len(df) == 0:
    print("All records have already been processed")
else:
    print(f"Starting to process {len(df)} records...")
    print(f"Writing JSON files to {output_path}...")
    print("\n" + "="*60 + "\n")

    for index, row in df.iterrows():
        row_dict = row.to_dict()
        row_dict['entity_id'] = row_dict['entity_id']
        print(f"Entire row being written: {row_dict}")

        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        filename = f"fraud_watchlist_{timestamp}_{index}.json"
        file_path = f"{output_path}/{filename}"

        json_content = json.dumps(row_dict)
        dbutils.fs.put(file_path, json_content, overwrite=True)
        
        print(f"Row {index + 1}/{len(df)} written {filename}")
        print(f"Watchlist ID: {row_dict.get('watchlist_id')}")

        if index < len(df) - 1:
            print(f"Waiting 5 seconds...\n")
            time.sleep(5)

    print("\n" + "="*60)
    print(f"\nCompleted! Generated {len(df)} JSON files in {output_path}")